In [ ]:
# Exercice 1 — Décorateur simple (≈ 15 min)
# Écrire un décorateur @log_calls qui imprime le nom de la fonction, ses arguments, et son retour à chaque appel.

# @log_calls
# def add(a, b):
#     return a + b

# add(2, 3)
# # Calling add(2, 3)
# # add returned 5

# Utiliser @functools.wraps. Vérifier que add.__name__ reste 'add'.

from functools import wraps

def log_calls(func):  # type: ignore
    @wraps(func)
    def wrapper(*args, **kwargs):  # type: ignore
        print(f"Calling {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@log_calls
def add(a: int, b: int) -> int:
    return a + b

add(2, 3)
print(f"\n{add.__name__=}")

In [ ]:
# Exercice 2 — Décorateur paramétré (≈ 25 min)
# Écrire un décorateur @validate(*types) qui vérifie le type de chaque argument positionnel :

# @validate(int, int)
# def add(a, b):
#     return a + b

# add(2, 3)            # 5
# add("a", "b")        # TypeError: expected int, got str
# Bonus : étendre @validate(*types, **kwtypes) pour gérer aussi les arguments nommés.

from functools import wraps

def validate(*types):  # type: ignore
    def decorator(func):  # type: ignore
        @wraps(func)
        def wrapper(*args, **kwargs): # type: ignore
            for (index, arg) in enumerate(args):
                if type(arg) != types[index]:
                    raise TypeError(f"expected {types[index]}, got {type(arg)}")
            result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@validate(int, int)
def add(a: int, b: int) -> int:
    return a + b

add(2, 3)
add("a", "b")

In [ ]:
# Exercice 3 — Classe comme décorateur (≈ 25 min)
# Écrire une classe RateLimiter utilisable comme décorateur,
#   qui limite une fonction à max_calls appels par fenêtre glissante de period secondes.

# @RateLimiter(max_calls=5, period=1.0)
# def api_call():
#     ...
# Au-delà de 5 appels par seconde, l'appel suivant doit lever RuntimeError("rate limit exceeded").

from functools import wraps
from time import time, sleep

RATE_LIMIT_PER_PERIOD = 5

class RateLimiter:
    max_calls: int
    period: float  # in seconds

    def __init__(self, max_calls, period):  # type: ignore
        self.last_calls = []
        self.max_calls = max_calls
        self.period = period

    def __call__(self, func):  # type: ignore
        @wraps(func)  # type: ignore
        def wrapper(*args, **kwargs):  # type: ignore
            now = int(time())
            self.last_calls.append(now)
            self.last_calls = [c for c in self.last_calls if now - c < RATE_LIMIT_PER_PERIOD]
            if (len(self.last_calls) > RATE_LIMIT_PER_PERIOD):
                raise RuntimeError("rate limit exceeded")
            return func(*args, **kwargs)  # type: ignore
        return wrapper

@RateLimiter(max_calls=5, period=1.0)
def api_call():
    ...

print("Trying 5 quick calls...")
for _ in range(5):
    api_call()
print("All good!")

print("\nResetting...\n")
sleep(5)

print("Trying 8 spaced calls...")
for _ in range(8):
    api_call()
    sleep(1)
print("All good!")

print("\nResetting...\n")
sleep(5)

print("Trying 6 rapid calls...")
for _ in range(6):
    api_call()
# throws

In [ ]:
# Exercice 4 — Empilement (≈ 20 min)
# Combiner @timer et @retry(times=3, exceptions=(ConnectionError,)) sur une même fonction.

# Tester d'abord @timer au-dessus de @retry : le timer inclut-il les retries ?
# Inverser l'ordre. Comparer les sorties.
# Documenter en commentaire quel ordre est généralement souhaité, et pourquoi.


from functools import wraps
from time import perf_counter, sleep

def timer(func):  # type: ignore
    @wraps(func)
    def wrapper(*args, **kwargs):  # type: ignore
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

def retry(times=3, exceptions=(Exception,)):  # type: ignore
    def decorator(func):  # type: ignore
        @wraps(func)
        def wrapper(*args, **kwargs):  # type: ignore
            last = None
            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last = e
            raise last  # type: ignore
        return wrapper
    return decorator

attempts = {"n": 0}

@retry(times=3)
@timer
def flaky_rt(_) -> str:
    attempts["n"] += 1
    sleep(1)
    if attempts["n"] < 3:
        raise ConnectionError("transient")
    return "ok"

@timer
@retry(times=3)
def flaky_tr(_) -> str:
    attempts["n"] += 1
    sleep(1)
    if attempts["n"] < 3:
        raise ConnectionError("transient")
    return "ok"

def main():
    start = perf_counter()
    try:
        flaky_rt(1)
    except Exception as e:
        print("Raised:", e)
    finally:
        elapsed = perf_counter() - start
        print(f"Total of {flaky_rt.__name__} took {elapsed:.4f}s")

    print()
    attempts["n"] = 0

    start = perf_counter()
    try:
        flaky_tr(1)
    except Exception as e:
        print("Raised:", e)
    finally:
        elapsed = perf_counter() - start
        print(f"Total of {flaky_tr.__name__} took {elapsed:.4f}s")

main()
# rt monitore la durée du premier appel qui fonctionne - faible
# tr monitore la durée du traitement total de l'appel, retry compris - fort

In [ ]:
# Exercice 5 — Trois paradigmes pour un même algorithme (≈ 30 min)
# Implémenter en trois versions une fonction qui prend une liste d'opérations bancaires
#   (chaque opération est un dict avec type parmi ["debit", "credit"] et amount) et renvoie le solde final :

# Version impérative (boucle, accumulateur).
# Version fonctionnelle (functools.reduce ou expression en une ligne).
# Version objet (classe Account avec apply(operation) et balance).
# Comparer les trois sur trois axes : lisibilité, testabilité, extensibilité (par exemple : ajouter une commission par opération).

from typing import TypedDict, Union, Literal

class Operation(TypedDict):
    op_type: Union[Literal["debit"], Literal["credit"]]
    amount: float

class Account:
    final_amount: float

    def __init__(self):
        self.final_amount = 0

    def apply(self, op: Operation):
        self.final_amount += op["amount"] * (1 if op["op_type"] == "credit" else -1)

    def balance(self) -> float:
        return self.final_amount

CASES: list[tuple[str, list[Operation], float]] = [
    # (name, operations, expected_balance)
    ("empty",          [],                                                                    0),
    ("single credit",  [{"op_type": "credit", "amount": 100}],                                 100),
    ("single debit",   [{"op_type": "debit",  "amount": 30}],                                  -30),
    ("mixed",          [{"op_type": "credit", "amount": 100},
                        {"op_type": "debit",  "amount": 40},
                        {"op_type": "credit", "amount": 10}],                                    70),
    ("nets to zero",   [{"op_type": "credit", "amount": 50},
                        {"op_type": "debit",  "amount": 50}],                                     0),
    ("goes negative",  [{"op_type": "debit",  "amount": 20},
                        {"op_type": "credit", "amount": 5}],                                    -15),
    ("floats",         [{"op_type": "credit", "amount": 0.1},
                        {"op_type": "credit", "amount": 0.2}],                                   0.3),
    ("zero amounts",   [{"op_type": "credit", "amount": 0},
                        {"op_type": "debit",  "amount": 0}],                                      0),
    ("many ops",       [{"op_type": "credit", "amount": i} for i in range(100)],             4950),
]

def list_operations_imperative(operations: list[Operation]) -> float:
    final_amount = 0.0
    for op in operations:
        final_amount += op["amount"] * (1 if op["op_type"] == "credit" else -1)
    return final_amount

def list_operations_functional(operations: list[Operation]) -> float:
    return sum([op["amount"] * (1 if op["op_type"] == "credit" else -1) for op in operations])

def run_all(func):  # type: ignore
    for name, operations, expected in CASES:
        result = round(func(operations), 5)
        assert result == expected, f"Test case '{name}' failed: expected {expected}, got {result}"
    print(f"All test cases passed for {func.__name__}.")

def run_all_obj(func):  # type: ignore
    for name, operations, expected in CASES:
        account = Account()
        for op in operations:
            account.apply(op)
        result = round(account.balance(), 5)
        assert result == expected, f"Test case '{name}' failed: expected {expected}, got {result}"
    print(f"All test cases passed for {func.__name__}.")

run_all(list_operations_imperative)
run_all(list_operations_functional)
run_all_obj(Account)

# Lisibilité : Objet clair vainqueur, Fonctionnel clair perdant
# Testabilité : Objet clair vainqueur
# Extensibilité : Objet clair vainqueur, Fonctionnel clair perdant

In [ ]:
# 10. Mini-défi de synthèse (≈ 1 à 2 heures)
# Concevoir un système de plugins paramétrés via décorateurs :

# @plugin(name="uppercase", priority=10)
# def to_upper(text):
#     return text.upper()

# @plugin(name="reverse", priority=5)
# def reverse(text):
#     return text[::-1]
# @plugin(...) enregistre la fonction dans un registre global (plugin_registry) avec son nom et sa priorité.

# build_pipeline() renvoie une fonction qui applique tous les plugins en chaîne, dans l'ordre de priorité décroissante.

# pipeline = build_pipeline()
# pipeline("hello")   # "OLLEH" — uppercase (priority 10), puis reverse (priority 5)
# Bonus : ajouter un décorateur @traced qui log chaque étape (empilable avec @plugin).

# @traced
# @plugin(name="uppercase", priority=10)
# def to_upper(text):
#     return text.upper()
# Validation : pipeline("hello") doit produire "OLLEH" et logger l'entrée/sortie de chaque plugin si @traced est appliqué.

from functools import wraps

plugin_registry = {}

def traced(func):  # type: ignore
    @wraps(func)
    def wrapper(*args, **kwargs):  # type: ignore
        print(f"Entering {func.__name__} with args: {args}, kwargs: {kwargs}")
        result = func(*args, **kwargs)
        print(f"Exiting {func.__name__} with result: {result}")
        return result
    return wrapper

def plugin(name, priority):  # type: ignore
    def decorator(func):  # type: ignore
        plugin_registry[name] = {  # type: ignore
            'function': func,
            'priority': priority
        }
        return func
    return decorator

@plugin(name="uppercase", priority=10)
@traced
def to_upper(text):  # type: ignore
    return text.upper()

@plugin(name="reverse", priority=5)
@traced
def reverse(text):  # type: ignore
    return text[::-1]

def build_pipeline():  # type: ignore
    def pipeline(text):  # type: ignore
        # Trier les plugins par priorité décroissante
        sorted_plugins = sorted(plugin_registry.values(), key=lambda x: x['priority'], reverse=True)
        for plugin in sorted_plugins:
            text = plugin['function'](text)
        print(text)
    return pipeline

pipeline = build_pipeline()
pipeline("hello")   # "OLLEH" — uppercase (priority 10), puis reverse (priority 5)

Le module M6 est validé lorsque :

- [x] L'apprenant peut expliquer ce qu'est un décorateur en une phrase, avec une analogie.
- [x] Il sait écrire un décorateur simple, paramétré, et basé sur une classe.
- [x] Il utilise @functools.wraps systématiquement.
- [x] Il peut prédire le comportement d'une pile de décorateurs avant exécution.
- [x] Il peut citer 3 décorateurs standards de la stdlib et leur usage.
- [x] Il distingue les trois paradigmes et peut implémenter le même algorithme dans deux d'entre eux.
- [x] Il peut justifier son choix de paradigme sur un problème donné, sans dogmatisme.